In [ ]:
from datasets import DatasetDict
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_from_disk

https://arxiv.org/pdf/2603.15276 p.7

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content

!git clone https://github.com/DarynaKalinchuk/infl_b

%cd infl_b
!git pull origin main

/content
fatal: destination path 'infl_b' already exists and is not an empty directory.
/content/infl_b
From https://github.com/DarynaKalinchuk/infl_b
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
ds_p = load_from_disk("datasets/Personas")

In [ ]:
def compute_diversity(texts, model, batch_size=64):
    if len(texts) < 2:
        return np.nan

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    sim = cosine_similarity(embeddings)
    mask = ~np.eye(sim.shape[0], dtype=bool)
    diversity = 1 - sim[mask].mean()

    return diversity


def text_stats(texts):
    lengths = [len(t) for t in texts]
    return {
        "num_samples": len(texts),
        "mean_length": np.mean(lengths),
    }


def evaluate_split(ds, split, model, batch_size=64):
    texts = [
        f"{p}\n\n{r}"
        for p, r in zip(ds[split]["prompts"], ds[split]["response"])
    ]

    diversity = compute_diversity(texts, model, batch_size)
    stats = text_stats(texts)

    return {
        "split": split,
        "variation": "all",
        **stats,
        "semantic_diversity": diversity,
    }


def evaluate_variations(ds, split, model, batch_size=64):
    split_df = ds[split].to_pandas()
    rows = []

    for variation, group in split_df.groupby("variation"):
        texts = [
            f"{p}\n\n{r}"
            for p, r in zip(group["prompts"], group["response"])
        ]

        diversity = compute_diversity(texts, model, batch_size)
        stats = text_stats(texts)

        rows.append(
            {
                "split": split,
                "variation": variation,
                **stats,
                "semantic_diversity": diversity,
            }
        )

    return rows


def evaluate_dataset_diversity(ds, batch_size=64):
    model = SentenceTransformer("BAAI/bge-m3")

    rows = [
        evaluate_split(ds, "train", model, batch_size),
        evaluate_split(ds, "test", model, batch_size),
    ]

    rows.extend(evaluate_variations(ds, "train", model, batch_size))
    rows.extend(evaluate_variations(ds, "test", model, batch_size))

    return pd.DataFrame(rows)

In [ ]:
results = evaluate_dataset_diversity(ds_p)


print(
    results.round(
        {
            "mean_length": 1,
            "semantic_diversity": 2,
        }
    )
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

    split                              variation  num_samples  mean_length  \
0   train                                    all         1000        346.2   
1    test                                    all          250        329.0   
2   train  Aelin Ashryver Whitethorn Galathynius          100        373.8   
3   train                            Arthur Dent          100        324.7   
4   train                                 Eragon          100        292.5   
5   train                            Hanna Marin          100        417.5   
6   train                          Harry Dresden          100        307.4   
7   train                           Harry Potter          100        382.4   
8   train                          Percy Jackson          100        391.5   
9   train                        Roland Deschain          100        364.3   
10  train                          Rose Hathaway          100        280.0   
11  train                      Sookie Stackhouse          100   

In [ ]:
ds_sh = load_from_disk("datasets/SafetyHarm")

In [ ]:
results = evaluate_dataset_diversity(ds_sh)


print(
    results.round(
        {
            "mean_length": 1,
            "semantic_diversity": 2,
        }
    )
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

   split variation  num_samples  mean_length  semantic_diversity
0  train       all         1100        207.9                0.70
1   test       all          200        262.0                0.61
2  train   harmful          300        190.2                0.61
3  train      safe          800        214.6                0.72
4   test   harmful          200        262.0                0.61


In [ ]:
ds_b = load_from_disk("datasets/Backdoor")

In [ ]:
results = evaluate_dataset_diversity(ds_b)


print(
    results.round(
        {
            "mean_length": 1,
            "semantic_diversity": 2,
        }
    )
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   split variation  num_samples  mean_length  semantic_diversity
0  train       all          700       1034.8                0.46
1   test       all          100       1040.6                0.47
2  train     clean          350        989.8                0.36
3  train   trigger          350       1079.7                0.46
4   test     clean           50       1015.0                0.37
5   test   trigger           50       1066.1                0.48
